In [ ]:
import os
import gc
import re
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr

In [ ]:
pip install netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.4 MB/s eta 0:00:00


In [ ]:
file_path = "/content/drive/MyDrive/Project Ignis/CMIP/Historical/tasmax_day_ACCESS-ESM1-5_historical_r1i1p1f1_gn_1990_v2.0.nc"

ds = xr.open_dataset(file_path)

In [ ]:
ds


<xarray.Dataset> Size: 1GB
Dimensions:  (time: 365, lat: 600, lon: 1440)
Coordinates:
  * time     (time) datetime64[ns] 3kB 1990-01-01T12:00:00 ... 1990-12-31T12:...
  * lat      (lat) float64 5kB -59.88 -59.62 -59.38 -59.12 ... 89.38 89.62 89.88
  * lon      (lon) float64 12kB 0.125 0.375 0.625 0.875 ... 359.4 359.6 359.9
Data variables:
    tasmax   (time, lat, lon) float32 1GB ...
Attributes: (12/21)
    activity:              NEX-GDDP-CMIP6
    Conventions:           CF-1.7
    frequency:             day
    institution:           NASA Earth Exchange, NASA Ames Research Center, Mo...
    variant_label:         r1i1p1f1
    product:               output
    ...                    ...
    cmip6_institution_id:  CSIRO
    cmip6_license:         CC-BY-SA 4.0
    contact:               Dr. Bridget Thrasher: bridget@climateanalyticsgrou...
    creation_date:         Sat Nov 16 13:37:04 PST 2024
    disclaimer:            These data are considered provisional and subject ...
    tracking_id:           683d9f1e-d267-4902-a1fe-3974f652119c

Dont forget to remove ## if it is in kelvinConvert Kelvin → Celsius

In [ ]:
import os
import xarray as xr
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

def extract_city_cmip(
    nc_path,
    csv_path,
    varname="tasmax",
    method="interp",
    parquet_path=None
):

    ds = xr.open_dataset(nc_path)
    da = ds[varname]
    lon_coords = ds["lon"].values # Storing actual longitude coordinates
    lat_coords = ds["lat"].values  # Storing actual latitude coordinates

    cities = pd.read_csv(csv_path)
    # assume first three cols are country, lat, lon (adapt if different)
    cities = cities.rename(columns={cities.columns[0]: "country",
                                    cities.columns[1]: "lat",
                                    cities.columns[2]: "lon"})
    cities = cities.reset_index(drop=True)

    # Fix longitude convention for cities data
    if lon_coords.min() >= 0 and lon_coords.max() > 180:
        cities["lon_mod"] = cities["lon"] % 360
    else:
        cities["lon_mod"] = ((cities["lon"] + 180) % 360) - 180

    # ---------- Interpolation ----------
    if method == "interp":
        interp = da.interp(
            lat=xr.DataArray(cities["lat"].values, dims="points"),
            lon=xr.DataArray(cities["lon_mod"].values, dims="points"),
            method="linear",
        )
        data = interp.values  # shape: (time, n_cities)

    # ---------- Nearest ----------
    else:
        LON, LAT = np.meshgrid(lon_coords, lat_coords)
        pts = np.vstack([LAT.ravel(), LON.ravel()]).T
        tree = cKDTree(pts)

        city_pts = np.vstack([cities["lat"].values, cities["lon_mod"].values]).T
        _, idx = tree.query(city_pts)

        lat_len, lon_len = len(lat_coords), len(lon_coords)
        ilat = idx // lon_len
        ilon = idx % lon_len

        data = np.stack([da[:, ilat[i], ilon[i]].values for i in range(len(cities))], axis=1)

    # ------------------------------------------------------
    # Convert to tidy long-format:
    # time | country | lat | lon | tasmax_C
    # ------------------------------------------------------
    # Use the numerical index of the 'cities' DataFrame as column names to maintain uniqueness
    df = pd.DataFrame(data, columns=cities.index, index=pd.to_datetime(ds["time"].values)) # Use 'time' from ds
    df = df.reset_index().rename(columns={"index": "time"})

    long_df = df.melt(
        id_vars="time",
        var_name="city_original_index", # Store the original index for mapping
        value_name="tasmax_K"
    )

    # Map back country, lat, and lon using the unique city_original_index
    long_df["country"] = long_df["city_original_index"].map(cities["country"])
    long_df["lat"] = long_df["city_original_index"].map(cities["lat"])
    long_df["lon"] = long_df["city_original_index"].map(cities["lon"])

    # Convert Kelvin → Celsius
    long_df["tasmax_C"] = long_df["tasmax_K"] - 273.15

    # Drop Kelvin column and the temporary city_original_index column
    long_df = long_df[["time", "country", "lat", "lon", "tasmax_C"]]

    # ------------------------------------------------------
    # Save parquet
    # ------------------------------------------------------
    if parquet_path:
        long_df.to_parquet(parquet_path, compression="brotli")
        print(f"Saved tidy parquet → {parquet_path}")

    return long_df

In [ ]:
def process_all_cmip_files(
    folder_path,
    csv_path,
    varname="tasmax",
    method="interp",
    out_folder=None
):
    if out_folder is None:
        out_folder = folder_path  # save output in same folder

    os.makedirs(out_folder, exist_ok=True)

    # List all NetCDF files
    nc_files = [f for f in os.listdir(folder_path) if f.endswith(".nc")]

    print(f"Found {len(nc_files)} NetCDF files.")

    for nc_file in nc_files:
        nc_path = os.path.join(folder_path, nc_file)

        # Try to infer year from filename (optional but helpful)
        year = None
        for token in nc_file.split("_"):
            if token.isdigit() and len(token) == 4:
                year = token

        out_name = f"{varname}_{year if year else 'unknown'}_{os.path.splitext(nc_file)[0]}.parquet"
        out_path = os.path.join(out_folder, out_name)

        print(f"Processing: {nc_file} → {out_name}")

        extract_city_cmip(
            nc_path=nc_path,
            csv_path=csv_path,
            varname=varname,
            method=method,
            parquet_path=out_path
        )

    print("✔ All files processed.")


In [ ]:
process_all_cmip_files(
    folder_path="/content/drive/MyDrive/Project Ignis/CMIP/2024-2025", # folder containing *.nc CMIP6 files
    csv_path="/content/drive/MyDrive/Project Ignis/gadb_country_declatlon.csv", # your 4000+ city coordinates
    varname="tasmax",
    method="interp",
    out_folder="/content/drive/MyDrive/Project Ignis/CMIP/2024-2025/Compress"
)

Found 13 NetCDF files.
Processing: tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2023_v2.0.nc → tasmax_2023_tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2023_v2.0.parquet
Saved tidy parquet → /content/drive/MyDrive/Project Ignis/CMIP/2024-2025/Compress/tasmax_2023_tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2023_v2.0.parquet
Processing: tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2024_v2.0.nc → tasmax_2024_tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2024_v2.0.parquet
Saved tidy parquet → /content/drive/MyDrive/Project Ignis/CMIP/2024-2025/Compress/tasmax_2024_tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2024_v2.0.parquet
Processing: tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2020_v2.0.nc → tasmax_2020_tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2020_v2.0.parquet
Saved tidy parquet → /content/drive/MyDrive/Project Ignis/CMIP/2024-2025/Compress/tasmax_2020_tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2020_v2.0.parquet
Processing: tasmax_day_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_2015_v2.0.nc → tas